<a href="https://colab.research.google.com/github/err0rgod/go-code/blob/main/semantic_search_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tasks-
Split text
create embeddigs
store embeddings in a numpy array
embed the user's querry
calculate cosine similarity
return top 3 chunks


go from normal text docs to pdf's and then a entire data


In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import snapshot_download

docs_path = snapshot_download(
    repo_id="err0rgod/india_history_semantic_search",
    repo_type="dataset",
    local_dir="/content/semantic-search-docs"
)

print("Documents downloaded to:", docs_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Documents downloaded to: /content/semantic-search-docs


In [ ]:
!pip install -q sentence-transformers langchain-text-splitters

In [ ]:
# filename = docs_path + "/01_indus_valley.md"

# with open(filename, "r", encoding="utf-8") as f:
#     text = f.read()


# load multiple files
import os
import  glob
import numpy as np


# break in chunks
def chunk_text(text, chunk_size=100,overlap=20):
  words = text.split() #break the string into words
  chunks = []
  step = chunk_size-overlap #how far we sslide the window each time

  for start in range(0, len(words), step):
    chunk_words = words[start:start+chunk_size]
    if not chunk_words:
      break
    chunks.append(" ".join(chunk_words))
    if start + chunk_size >= len(words):
      break
  return chunks



# create embedding from chunks
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
DATA_PATH = "/content/semantic-search-docs"
all_chunks = []
metadata = []
for filepath in glob.glob(os.path.join(DATA_PATH, "*.md")):
  with open(filepath, "r", encoding="utf-8") as f:
    text = f.read()
  chunks = chunk_text(text)
  for i,chunk in enumerate(chunks):
    all_chunks.append(chunk)
    metadata.append({"source": filepath, "chunk_index": i})

chunk_embeddings = model.encode(all_chunks, normalize_embeddings=True)
print(chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(49, 384)


In [ ]:
# take user input and convert into embeddings
query = "In which age indus valley was."
query_embedding = model.encode(query, normalize_embeddings=True)

# calculate cosine similarity
scores =  chunk_embeddings @ query_embedding
print(scores)

# return top k chunks
import numpy as np
top_k = 2
top_indices = np.argsort(-scores)[:top_k]   # sort descending, take top 3

for idx in top_indices:
    print(f"score={scores[idx]:.4f}")
    print(all_chunks[idx])
    print("---")

[ 0.1572828   0.23845488  0.17999491  0.7114569   0.36530143  0.24652392
  0.15502185  0.0857313   0.17430127  0.31824183  0.23370863  0.32760322
  0.24048683  0.25433624  0.2850671   0.19428161  0.240275    0.25541884
  0.05923091 -0.00602956 -0.02903219  0.33903697  0.35522187  0.41382492
  0.30130762  0.3361382   0.31389126  0.130839    0.13889879  0.14762115
 -0.05891718  0.42218173  0.19710511  0.3922573   0.2607252   0.09186435
  0.21087924  0.3966062   0.33914858  0.3351174   0.25509554  0.30798662
  0.3513273   0.32134667  0.35938382  0.25102156  0.14592242  0.19011244
  0.1743345 ]
score=0.7115
# Indus Valley Civilization ## Overview The Indus Valley Civilization (IVC) was a prominent Bronze Age civilization in the northwestern regions of South Asia, lasting from approximately 3300 BCE to 1300 BCE. It flourished primarily in the basins of the Indus River and the Ghaggar-Hakra River. Along with Ancient Egypt and Mesopotamia, it was one of three early civilizations of the Near E